# Сравнение DLightSB и DLightSB-MM

Ноутбук принимает один ZIP-архив пары экспериментов созданный ноутбуком `run_dlight_sb_pairs_colab.ipynb`

Поддерживаются архивы для размерностей 2 16 и 64


## Импорты и оформление


In [ ]:
from pathlib import Path
import json
import shutil
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from IPython.display import display

plt.style.use("default")
plt.rcParams.update(
    {
        "figure.figsize": (9, 5),
        "axes.grid": True,
        "grid.alpha": 0.25,
        "lines.linewidth": 2,
        "lines.markersize": 6,
    }
)


## Загрузка архива

Выберите ровно один ZIP-файл с парой экспериментов


In [ ]:
from google.colab import files

uploaded = files.upload()
archive_names = [
    name
    for name in uploaded
    if name.lower().endswith(".zip")
]

if len(archive_names) != 1:
    raise ValueError(
        f"Нужно загрузить один ZIP-архив получено {len(archive_names)}"
    )

ARCHIVE_PATH = Path(archive_names[0]).resolve()
print("Загружен архив:", ARCHIVE_PATH.name)


## Распаковка и чтение результатов


In [ ]:
ANALYSIS_ROOT = Path("/content/catsbench_pair_analysis")
EXTRACT_DIR = ANALYSIS_ROOT / ARCHIVE_PATH.stem

if EXTRACT_DIR.is_dir():
    shutil.rmtree(EXTRACT_DIR)

EXTRACT_DIR.mkdir(parents=True)

with zipfile.ZipFile(ARCHIVE_PATH) as archive:
    root = EXTRACT_DIR.resolve()

    for member in archive.infolist():
        target = (EXTRACT_DIR / member.filename).resolve()

        if not target.is_relative_to(root):
            raise ValueError(
                f"Некорректный путь внутри архива: {member.filename}"
            )

    archive.extractall(EXTRACT_DIR)

print("Архив распакован в:", EXTRACT_DIR)


In [ ]:
MODEL_DIRECTORIES = {
    "Gradient descent": "baseline",
    "MM": "mm",
}

MODEL_COLORS = {
    "Gradient descent": "#1f77b4",
    "MM": "#ff7f0e",
}


def find_one(run_dir: Path, pattern: str) -> Path:
    matches = sorted(run_dir.glob(pattern))

    if len(matches) != 1:
        raise FileNotFoundError(
            f"Для {pattern} ожидался один файл найдено {len(matches)}"
        )

    return matches[0]


def read_run(run_dir: Path) -> dict:
    train_metrics_path = find_one(run_dir, "*/train/metrics.csv")
    test_metrics_path = find_one(run_dir, "*/test/metrics.csv")
    hparams_path = find_one(run_dir, "*/train/hparams.yaml")

    with hparams_path.open(encoding="utf-8") as file:
        config = yaml.safe_load(file)

    return {
        "run_dir": run_dir,
        "train": pd.read_csv(train_metrics_path),
        "test": pd.read_csv(test_metrics_path),
        "config": config,
    }


with (EXTRACT_DIR / "manifest.json").open(encoding="utf-8") as file:
    manifest = json.load(file)

runs = {
    label: read_run(EXTRACT_DIR / directory)
    for label, directory in MODEL_DIRECTORIES.items()
}

print("Размерность:", manifest["dimension"])
print("Seed:", manifest["seed"])
print("Commit:", manifest["commit"])


## Финальные метрики

Для score и effective k лучшим считается большее значение

Для loss и log c лучшим считается меньшее значение

Для log v лучшим считается большее значение поскольку `loss = log_c - log_v`


In [ ]:
SUMMARY_COLUMNS = [
    "cond_shape_score_forward",
    "cond_trend_score_forward",
    "loss",
    "log_c",
    "log_v",
    "posterior_eff_k",
    "prior_eff_k",
    "shape_score_forward",
    "trend_score_forward",
]

MINIMIZE_COLUMNS = {"loss", "log_c"}
BEST_COLOR = "#6f42c1"


def final_test_values(frame: pd.DataFrame) -> dict:
    values = {}

    for column in frame.columns:
        if not column.startswith("test/"):
            continue

        series = frame[column].dropna()

        if len(series):
            values[column.removeprefix("test/")] = float(series.iloc[-1])

    if "log_c" not in values or "log_v" not in values:
        raise KeyError("В test CSV отсутствуют log_c или log_v")

    values["loss"] = values["log_c"] - values["log_v"]
    return values


def highlight_best(frame: pd.DataFrame) -> pd.DataFrame:
    styles = pd.DataFrame("", index=frame.index, columns=frame.columns)

    for column in frame.columns:
        valid = frame[column].dropna()

        if valid.empty:
            continue

        best = valid.min() if column in MINIMIZE_COLUMNS else valid.max()
        winners = np.isclose(frame[column], best, rtol=1e-9, atol=1e-12)
        styles.loc[winners, column] = (
            f"background-color: {BEST_COLOR}; color: white; font-weight: bold"
        )

    return styles


In [ ]:
summary = pd.DataFrame(
    {
        label: final_test_values(run["test"])
        for label, run in runs.items()
    }
).T

missing_columns = [
    column
    for column in SUMMARY_COLUMNS
    if column not in summary.columns
]

if missing_columns:
    raise KeyError(
        "В итоговых логах отсутствуют метрики: "
        + ", ".join(missing_columns)
    )

summary = summary[SUMMARY_COLUMNS]

display(
    summary.style
    .format(precision=5, na_rep="—")
    .apply(highlight_best, axis=None)
    .set_caption("Финальные test-метрики")
)


## История validation-метрик по объёму обработанных данных

Точки располагаются по фактическому числу объектов обработанных к моменту validation

Ось объёма данных логарифмическая поскольку бюджеты baseline и MM отличаются на несколько порядков


In [ ]:
VALIDATION_COLUMNS = [
    "val/log_c",
    "val/log_v",
    "val/cond_shape_score_forward",
    "val/cond_trend_score_forward",
]


def last_non_null(series: pd.Series):
    values = series.dropna()
    return values.iloc[-1] if len(values) else np.nan


def training_progress(run: dict) -> pd.DataFrame:
    frame = run["train"]
    columns = [
        "train/time",
        "train/batches",
        "train/samples",
        "train/device/samples_per_sec",
    ]
    missing = [
        column
        for column in ["step", *columns[:3]]
        if column not in frame.columns
    ]

    if missing:
        raise KeyError(
            "В train CSV отсутствуют метрики ThroughputMonitor: "
            + ", ".join(missing)
        )

    available = [column for column in columns if column in frame.columns]
    progress = (
        frame[["step", *available]]
        .dropna(subset=["train/time", "train/samples"])
        .groupby("step", as_index=False)
        .agg({column: last_non_null for column in available})
        .sort_values("train/samples")
        .reset_index(drop=True)
    )
    return progress


def validation_history(run: dict) -> pd.DataFrame:
    frame = run["train"]
    missing = [
        column
        for column in ["step", *VALIDATION_COLUMNS]
        if column not in frame.columns
    ]

    if missing:
        raise KeyError(
            "В train CSV отсутствуют validation-колонки: "
            + ", ".join(missing)
        )

    history = (
        frame[["step", *VALIDATION_COLUMNS]]
        .dropna(subset=VALIDATION_COLUMNS, how="all")
        .groupby("step", as_index=False)
        .agg({column: last_non_null for column in VALIDATION_COLUMNS})
        .sort_values("step")
        .reset_index(drop=True)
    )

    batch_size = int(run["config"]["data"]["batch_size"])
    accumulate = int(
        run["config"].get("trainer", {}).get("accumulate_grad_batches", 1)
    )

    history["expected_samples"] = (
        (history["step"].astype(int) + 1)
        * batch_size
        * accumulate
    )
    history["loss"] = history["val/log_c"] - history["val/log_v"]

    progress = training_progress(run).copy()
    history["expected_samples"] = history["expected_samples"].astype(float)
    progress["train/samples"] = progress["train/samples"].astype(float)
    history = pd.merge_asof(
        history.sort_values("expected_samples"),
        progress,
        left_on="expected_samples",
        right_on="train/samples",
        direction="nearest",
        suffixes=("", "_train"),
    )
    history["samples_seen"] = history["train/samples"].astype(int)
    history["train_time_seconds"] = history["train/time"]
    return history


In [ ]:
histories = {
    label: validation_history(run)
    for label, run in runs.items()
}

for label, history in histories.items():
    print(
        label,
        f"validation points={len(history)}",
        f"samples={history['samples_seen'].min():,}..{history['samples_seen'].max():,}",
    )


In [ ]:
plots = [
    ("loss", "Validation loss"),
    ("val/cond_shape_score_forward", "Conditional shape score"),
    ("val/cond_trend_score_forward", "Conditional trend score"),
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for axis, (column, title) in zip(axes, plots):
    for label, history in histories.items():
        points = history.dropna(subset=[column])
        axis.plot(
            points["samples_seen"],
            points[column],
            marker="o",
            color=MODEL_COLORS[label],
            label=label,
        )

    axis.set_xscale("log")
    axis.set_title(title)
    axis.set_xlabel("Обработано объектов")
    axis.legend()

axes[0].set_ylabel("Значение метрики")
fig.suptitle(
    f"DLightSB и DLightSB-MM d={manifest['dimension']}",
    fontsize=14,
)
plt.tight_layout()
plt.show()


## История validation-метрик по чистому времени обучения

`train/time` формируется callback `ThroughputMonitor` и не включает время validation


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for axis, (column, title) in zip(axes, plots):
    for label, history in histories.items():
        points = history.dropna(subset=[column, "train_time_seconds"])
        axis.plot(
            points["train_time_seconds"],
            points[column],
            marker="o",
            color=MODEL_COLORS[label],
            label=label,
        )

    axis.set_title(title)
    axis.set_xlabel("Чистое время обучения секунд")
    axis.legend()

axes[0].set_ylabel("Значение метрики")
fig.suptitle(
    f"Метрики по времени обучения d={manifest['dimension']}",
    fontsize=14,
)
plt.tight_layout()
plt.show()


## Чистое время обучения по объёму обработанных данных


In [ ]:
progress = {
    label: training_progress(run)
    for label, run in runs.items()
}

fig, axis = plt.subplots(figsize=(9, 5))

for label, frame in progress.items():
    axis.plot(
        frame["train/samples"],
        frame["train/time"],
        marker="o",
        color=MODEL_COLORS[label],
        label=label,
    )

axis.set_xscale("log")
axis.set_xlabel("Обработано объектов")
axis.set_ylabel("Чистое время обучения секунд")
axis.set_title("Время обучения по объёму данных")
axis.legend()
plt.tight_layout()
plt.show()


## Скорость обучения

Среднее время итерации и throughput рассчитаны из накопленного train-времени без validation


In [ ]:
performance_rows = {}

for label, frame in progress.items():
    final = frame.iloc[-1]
    seconds = float(final["train/time"])
    batches = int(final["train/batches"])
    samples = int(final["train/samples"])

    performance_rows[label] = {
        "train_time_seconds": seconds,
        "iterations": batches,
        "samples": samples,
        "seconds_per_iteration": seconds / batches,
        "samples_per_second": samples / seconds,
        "samples_per_minute": 60 * samples / seconds,
    }

performance = pd.DataFrame.from_dict(performance_rows, orient="index")
display(performance.style.format(precision=3).set_caption("Скорость обучения"))


## Использование памяти GPU во время обучения

Показана текущая память PyTorch CUDA allocator после train-update

Allocated соответствует памяти занятой тензорами

Reserved дополнительно включает кэш CUDA allocator


In [ ]:
GPU_ALLOCATED_COLUMN = (
    "DeviceStatsMonitor.on_train_batch_end/allocated_bytes.all.current"
)
GPU_RESERVED_COLUMN = (
    "DeviceStatsMonitor.on_train_batch_end/reserved_bytes.all.current"
)


def gpu_memory_history(run: dict) -> pd.DataFrame:
    frame = run["train"]
    required = ["step", GPU_ALLOCATED_COLUMN, GPU_RESERVED_COLUMN]
    missing = [column for column in required if column not in frame.columns]

    if missing:
        raise KeyError(
            "В train CSV отсутствуют GPU memory-метрики: "
            + ", ".join(missing)
        )

    memory = (
        frame[required]
        .dropna(subset=[GPU_ALLOCATED_COLUMN, GPU_RESERVED_COLUMN], how="all")
        .groupby("step", as_index=False)
        .agg(
            {
                GPU_ALLOCATED_COLUMN: last_non_null,
                GPU_RESERVED_COLUMN: last_non_null,
            }
        )
        .sort_values("step")
    )
    train_progress = training_progress(run).sort_values("step")
    memory = pd.merge_asof(
        memory,
        train_progress[["step", "train/samples"]],
        on="step",
        direction="nearest",
    )
    memory["allocated_mib"] = memory[GPU_ALLOCATED_COLUMN] / 1024**2
    memory["reserved_mib"] = memory[GPU_RESERVED_COLUMN] / 1024**2
    return memory


In [ ]:
memory_histories = {
    label: gpu_memory_history(run)
    for label, run in runs.items()
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for axis, (label, frame) in zip(axes, memory_histories.items()):
    axis.plot(
        frame["train/samples"],
        frame["allocated_mib"],
        label="Allocated",
        color=MODEL_COLORS[label],
    )
    axis.plot(
        frame["train/samples"],
        frame["reserved_mib"],
        label="Reserved",
        color=MODEL_COLORS[label],
        linestyle="--",
        alpha=0.8,
    )
    axis.set_xscale("log")
    axis.set_title(label)
    axis.set_xlabel("Обработано объектов")
    axis.set_ylabel("GPU memory MiB")
    axis.legend()

fig.suptitle("Память GPU во время обучения", fontsize=14)
plt.tight_layout()
plt.show()
